In [173]:
import os
import pandas as pd
import re
from pathlib import Path
import numpy as np
from datetime import datetime, timezone
import json

import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

CREDENTIALS_PATH = "credentials.json"
TOKEN_PATH = "token.json"

In [174]:
dirs = {
    "instruments_root": r"/Volumes/lab-windingm/data/instruments/behavioural_rigs/plugcamera/",
    "shared_root" :r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data",
    "local_root": r"/Users/meryv/repos/behavioural-rigs",

    "final-sheets": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/final-sheets", #these are view only
    "intermediate-sheets": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/intermediate-sheets", #these are not completed yet
    
    "mechano-screen" : r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/mechano-screen", #these are edited manually
    "split-screen" : r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/split-screen", 
    
    "backup-mechano": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/mechano-backup", #these are never touched 
    "backup-split": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/split-backup", 

    "plugcamera": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/plugcamera", #not used here
    "notebooks": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/notebooks", 
    
    "fly-lines": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/fly-lines" #sometimes gets updated on Dropbox
}

WEEK_TO_SPREADSHEET_ID = {
    2: "1Hzy2fWYd8pfVttOinSPZz3fxgQEqxnuFleBu1nYrR_E",
    11: "16SmOZUOif1WM1XIff_2Hor3ve8eANdYZVRxT5jHDssA", 
    3: "14I4pzm_53WucH9Ksl4VBrA3iIsCVpxUbMpeZXkSNXUg", 
    4: "10wWrUEvbP4WtVgj9NrPz2DuGSYkVU_vImNbWJcZnKDY"
}

SPREADSHEET_TO_EXP_NAME = {
    #date , first part of the weekly sheet name - type of screen
}


In [175]:
def get_gspread_client():
    creds = None
    
    if os.path.exists(TOKEN_PATH):
        creds = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                CREDENTIALS_PATH,
                SCOPES,
            )
            creds = flow.run_local_server(port=0)

        with open(TOKEN_PATH, "w") as token:
            token.write(creds.to_json())

    return gspread.authorize(creds)

def read_pupae_counts(experiment_name, nemo_root=dirs["instruments_root"]):
    predictions_dir = Path(nemo_root) / experiment_name / "pupae" / "predictions"

    if not predictions_dir.exists():
        raise FileNotFoundError(f"Predictions folder not found: {predictions_dir}")
    if not predictions_dir.is_dir():
        raise NotADirectoryError(f"Predictions path is not a folder: {predictions_dir}")

    csv_files = sorted(predictions_dir.glob("*.csv"))

    if len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV files found in: {predictions_dir}")

    return pd.read_csv(csv_files[0])

def standardise_missing_values(df):
    out = df.copy()

    missing_like = {
        "",
        "nan",
        "na",
        "n/a",
        "<na>",
        "none",
        "null",
        "nat",
    }

    for col in out.columns:
        if out[col].dtype == "object" or pd.api.types.is_string_dtype(out[col]):
            s = out[col].astype("string").str.strip()
            out[col] = out[col].mask(s.str.lower().isin(missing_like), pd.NA)

    return out


In [176]:
year = "2026"
target_week = 11

client = get_gspread_client()

spreadsheet_id = WEEK_TO_SPREADSHEET_ID[target_week]

spreadsheet = client.open_by_key(spreadsheet_id)
worksheet = spreadsheet.worksheet("Sheet1")

sheet_records = worksheet.get_all_records() #will not work with duplicate columns
sheet_df = pd.DataFrame(sheet_records)

print("Connected to:", spreadsheet.title)
#print("Worksheet:", worksheet.title)

sheet_df.head()

Connected to: 2026-04-06_mechano-screen_week-11


,experimenter,collector,plugcamera,condition,condition_comments,Other name,staging_date,date,time,amendments,...,weds_pm_removed_eggs_binary,thurs_am_mortality_n,thurs_am_removed_eggs_binary,thurs_pm_mortality_n,thurs_pm_removed_eggs_binary,fri_am_mortality_n,fri_am_removed_eggs_binary,fri_pm_mortality_n,fri_pm_removed_eggs_binary,last_edit
0,LK,HS,1.0,ss00886,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,LK,VM,85.0,ss01762,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,29-04-2026
2,LK,VM,3.0,ss52511,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,LK,HS,4.0,49828-1,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,LK,VM,5.0,ss01732,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,29-04-2026


In [177]:
match_cols = ["condition", "date", "time"]

s = sheet_df["staging_date"].astype("string").str.strip()
has_staging_date = s.notna() & s.ne("")

range_parts = s.str.extract(
    r"^\s*(?P<first_day>\d{1,2})\s*-\s*\d{1,2}/(?P<month>\d{1,2})/(?P<year>\d{4})\s*$"
)

first_date_from_range = (
    range_parts["first_day"].str.zfill(2)
    + "/"
    + range_parts["month"].str.zfill(2)
    + "/"
    + range_parts["year"]
)

first_full_date = s.str.extract(
    r"(?P<first_date>\d{1,2}/\d{1,2}/\d{4})",
    expand=False
)

first_date = first_date_from_range.where(
    range_parts["first_day"].notna(),
    first_full_date
)

parsed_date = pd.to_datetime(
    first_date,
    format="%d/%m/%Y",
    errors="coerce"
)

sheet_df["date"] = pd.NA
sheet_df.loc[has_staging_date, "date"] = parsed_date.dt.strftime("%d-%m")

sheet_df["date"] = sheet_df["date"].astype("string").str.strip()

if "time" not in sheet_df.columns:
        sheet_df["time"] = pd.NA
else: 
        print("Time and date colums already exist")

night_mask = s.str.contains(r"\d\s*-\s*\d", na=False)

sheet_df.loc[has_staging_date & night_mask, "time"] = "night"
sheet_df.loc[has_staging_date & ~night_mask, "time"] = "day"

#make sure they're clean
for col in match_cols:
    if col in sheet_df.columns:
        sheet_df[col] = (
            sheet_df[col]
            .astype("string")
            .str.replace("\u00a0", " ", regex=False)
            .str.strip()
            .str.lower()
        )

sheet_df

Time and date colums already exist


,experimenter,collector,plugcamera,condition,condition_comments,Other name,staging_date,date,time,amendments,...,weds_pm_removed_eggs_binary,thurs_am_mortality_n,thurs_am_removed_eggs_binary,thurs_pm_mortality_n,thurs_pm_removed_eggs_binary,fri_am_mortality_n,fri_am_removed_eggs_binary,fri_pm_mortality_n,fri_pm_removed_eggs_binary,last_edit
0,LK,HS,1.0,ss00886,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,LK,VM,85.0,ss01762,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,29-04-2026
2,LK,VM,3.0,ss52511,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,LK,HS,4.0,49828-1,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,LK,VM,5.0,ss01732,<NA>,<NA>,14-15/04/2026,14-04,night,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,29-04-2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,<NA>,HS,208.0,ss00918,<NA>,<NA>,17/04/2026,17-04,day,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,n/d,n/d,30-04-2026
116,<NA>,VM,209.0,ss55086,<NA>,<NA>,17/04/2026,17-04,day,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,n/d,n/d,29-04-2026
117,<NA>,HS,276.0,49766,<NA>,<NA>,17/04/2026,17-04,day,-1,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,n/d,n/d,<NA>
118,<NA>,VM,211.0,ss01989-2,<NA>,<NA>,17/04/2026,17-04,day,-1,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,n/d,n/d,<NA>


Load pupae batch

In [178]:
pupae_batch_folder = "VM_2026-04-29_mechano-screen-week-11_batch2_f11"

pupae_batch_csv = read_pupae_counts(pupae_batch_folder).copy() #this will only work if you've "connected" to the windingm folder by opening it in your laptop

pupae_batch_csv["pupae_source_row"] = pupae_batch_csv.index + 1
for d in [pupae_batch_csv]:
    d["pupae_batch_folder"] = pupae_batch_folder

pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].astype(str).str.strip() #making sure it's clean strings
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.split("predictions/", n=1).str[-1]
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.replace(
    r'(\d{1,2})([-_])(\d{1,2})([-_])(day|night)\b',
    r'\1-\2_\4',
    regex=True
)
pattern = r'(?i)(?P<experiment>.+?)[_-](?P<date>\d{1,2}-\d{2})[_-](?P<time>day|night)\_'
pupae_batch_csv[["condition", "date", "time"]] = pupae_batch_csv["dataset"].str.extract(pattern, expand=True)

#extract date of experiment
date_extracted = pupae_batch_csv["date"].str.extract(r'(\d{1,2}[-_]\d{1,2})', expand=False)
date_norm = date_extracted.str.replace("_", "-", regex=False)
#pupae_batch_csv["staging_date_clean_dt"] = pd.to_datetime(date_norm + f"-{year}", format="%d-%m-%Y", errors="coerce")

#calculate Pupae total
pupae_batch_csv["L_value"] = pupae_batch_csv["dataset"].str.extract(r'(?i)_L(\d+)', expand=False).astype("Int64")
pupae_batch_csv["Pupae"] = pupae_batch_csv["pupae_count"] + pupae_batch_csv["L_value"] #for clarity, the new original sheets should have a column called pupae_final rather than Pupae"

for col in match_cols:
    if col in pupae_batch_csv.columns:
        pupae_batch_csv[col] = (
            pupae_batch_csv[col]
            .astype("string")
            .str.replace("\u00a0", " ", regex=False)
            .str.strip()
            .str.lower()
        )

today = pd.Timestamp.today().strftime("%d-%m-%Y")
pupae_batch_csv["last_edit"] = today

print(
    pupae_batch_csv[
        [
            "pupae_source_row",
            "dataset",
            "condition",
            "date",
            "time",
            "Pupae",
            "pupae_count",
            "pupae_batch_folder",
        ]
    ].to_string(index=False)
)

 pupae_source_row                         dataset condition  date  time  Pupae  pupae_count                              pupae_batch_folder
                1   SS55086_16-04_day_L0.mp4.json   ss55086 16-04   day     86           86 VM_2026-04-29_mechano-screen-week-11_batch2_f11
                2 SS00886_15-04_night_L7.mp4.json   ss00886 15-04 night    159          152 VM_2026-04-29_mechano-screen-week-11_batch2_f11
                3 EMPTY-1_14-04_night_L6.mp4.json   empty-1 14-04 night    119          113 VM_2026-04-29_mechano-screen-week-11_batch2_f11
                4 SS55086_16-04_night_L4.mp4.json   ss55086 16-04 night    106          102 VM_2026-04-29_mechano-screen-week-11_batch2_f11
                5   ss27197_15-04_day_L2.mp4.json   ss27197 15-04   day     50           48 VM_2026-04-29_mechano-screen-week-11_batch2_f11
                6 SS27197_15-04_night_L0.mp4.json   ss27197 15-04 night     86           86 VM_2026-04-29_mechano-screen-week-11_batch2_f11
                7 SS

In [179]:
#Add safety information to the sheet - default do not merge automatically 

#are there any condition duplicates? do not merge automatically
raw_counts = (
    sheet_df.groupby(match_cols, dropna=False)
    .size()
    .rename("raw_sheet_matches")
    .reset_index()
)

sheet_df_wcounts = sheet_df.merge(
    raw_counts,
    on=match_cols,
    how="left"
)

#Leave the Pupae alone if the cell is not empty
pupae_s = sheet_df_wcounts["Pupae"].astype("string").str.strip()
pupae_empty = (
    sheet_df_wcounts["Pupae"].isna()
    | pupae_s.eq("")
    | pupae_s.str.lower().isin(["nan", "<na>", "none"])
)

#pupae_empty = (
    #sheet_df_wcounts["Pupae"].isna()
#)

sheet_df_wcounts["safe_for_pupae_merge"] = (
    sheet_df_wcounts["raw_sheet_matches"].eq(1)
    & pupae_empty
)
sheet_df_wcounts["pupae_merge_safety_reason"] = np.select(
    [
        sheet_df_wcounts["raw_sheet_matches"].gt(1),
        ~pupae_empty,
        sheet_df_wcounts["safe_for_pupae_merge"],
    ],
    [
        "duplicate condition/date/time in Google Sheet",
        "existing Pupae already present",
        "safe to merge",
    ],
    default="not safe to merge"
)

sheet_df_wcounts["safe_for_pupae_merge"] = (
    sheet_df_wcounts["safe_for_pupae_merge"]
    .fillna(False)
)
sheet_df_wcounts["pupae_merge_safety_reason"] = (
    sheet_df_wcounts["pupae_merge_safety_reason"]
    .fillna("not safe to merge")
)


sheet_df_wcounts = standardise_missing_values(sheet_df_wcounts)

sheet_df_wcounts_clean = sheet_df_wcounts[["condition", "date", "time", "raw_sheet_matches", "safe_for_pupae_merge", "pupae_merge_safety_reason"]].copy() #avoid column duplicates
sheet_df_wcounts_clean["pupae_merge_safety_reason"].value_counts()

sheet_df_wcounts["pupae_merge_safety_reason"].value_counts()

pupae_merge_safety_reason
existing Pupae already present                   59
safe to merge                                    49
duplicate condition/date/time in Google Sheet    12
Name: count, dtype: int64

In [180]:
sheet_safety_lookup = (
    sheet_df_wcounts_clean
    .groupby(match_cols, dropna=False)
    .agg(
        raw_sheet_matches=("raw_sheet_matches", "max"),
        safe_for_pupae_merge=("safe_for_pupae_merge", "any"),
        pupae_merge_safety_reason=(
            "pupae_merge_safety_reason",
            lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
        ),
    )
    .reset_index()
)
# rows are dropped because the duplicate conditions become one - they're still void anyway
sheet_safety_lookup["safe_for_pupae_merge"].value_counts()

safe_for_pupae_merge
False    65
True     49
Name: count, dtype: Int64

In [181]:
#this is the helpful information for pupae_batch_csv ... 
pupae_diag = pupae_batch_csv.merge(
    sheet_safety_lookup, 
    on=match_cols, 
    how="left", 
    validate="m:1"
)
pupae_diag["raw_sheet_matches"] = (
    pupae_diag["raw_sheet_matches"]
    .fillna(0)
    .astype(int)
)

pupae_diag["safe_for_pupae_merge"] = (
    pupae_diag["safe_for_pupae_merge"]
    .fillna(False)
)

pupae_diag["pupae_merge_safety_reason"] = (
    pupae_diag["pupae_merge_safety_reason"]
    .fillna("condition/date/time not found in Google Sheet")
)


In [182]:
pupae_for_merge = pupae_diag.loc[
    pupae_diag["safe_for_pupae_merge"]
].copy()

n_pupae = len(pupae_batch_csv)
n_pupae_safe_match = len(pupae_for_merge)
print(f"Pupae rows with safe sheet match: {n_pupae_safe_match}/{n_pupae}")

if n_pupae_safe_match == n_pupae:
    print("All pupae counts matched successfully.")
else:
    print(f"{n_pupae - n_pupae_safe_match} pupae rows did not match automatically.")

merged = sheet_df_wcounts.merge(
    pupae_for_merge,
    on=match_cols,
    how="left",
    suffixes=("", "_from_pupae"),
    indicator="merge_status",
)

matched_pupae_row = merged["merge_status"].eq("both") 

merged.loc[matched_pupae_row, "Pupae"] = merged.loc[
    matched_pupae_row,
    "Pupae_from_pupae"
]
merged.loc[matched_pupae_row, "pupae_batch_folder"] = merged.loc[
    matched_pupae_row,
    "pupae_batch_folder_from_pupae"
]

if "last_edit_from_pupae" in merged.columns:
    merged.loc[matched_pupae_row, "last_edit"] = merged.loc[
        matched_pupae_row,
        "last_edit_from_pupae"
    ]

merged[["condition", "date", "time", "Pupae", "Pupae_from_pupae", "dataset", "last_edit", "pupae_batch_folder"]].head(30)

Pupae rows with safe sheet match: 1/11
10 pupae rows did not match automatically.


,condition,date,time,Pupae,Pupae_from_pupae,dataset,last_edit,pupae_batch_folder
0,ss00886,14-04,night,<NA>,<NA>,NaN,<NA>,NaN
1,ss01762,14-04,night,<NA>,<NA>,NaN,29-04-2026,VM_2026-04-29_mechano-screen-week-11_batch3_f17
2,ss52511,14-04,night,<NA>,<NA>,NaN,<NA>,NaN
3,49828-1,14-04,night,<NA>,<NA>,NaN,<NA>,NaN
4,ss01732,14-04,night,<NA>,<NA>,NaN,29-04-2026,VM_2026-04-29_mechano-screen-week-11_batch3_f17
5,49766,14-04,night,140,<NA>,NaN,<NA>,VM_2026-04-29_mechano-screen-week-11_manual_f28
6,26259,14-04,night,0.0,<NA>,NaN,<NA>,VM_2026-04-27_mechano-week-11_manual_f17
7,ss38410,14-04,night,<NA>,<NA>,NaN,<NA>,NaN
8,empty-1,14-04,night,121.0,<NA>,NaN,29-04-2026,VM_2026-04-28_mechano-screen-week-11_batch1_f10
9,ss00886-2,14-04,night,<NA>,<NA>,NaN,<NA>,NaN


In [183]:
#add the bad ones at the bottom

pupae_unmatched = pupae_diag.loc[
    ~pupae_diag["safe_for_pupae_merge"]
].copy()

final_df = pd.concat(
    [merged, pupae_unmatched], 
    ignore_index=True
)


In [ ]:
drop_cols = [
    "merge_status",
    "raw_sheet_matches", 
    "safe_for_pupae_merge",
    "pupae_merge_safety_reason", 
    "_merge", 
    "L_value", 
    "raw_sheet_matches", 
    "Pupae_batch_folder_from_pupae", 
    "Pupae_from_pupae",
    "last_edit_from_pupae", 
    "safe_for_pupae_merge_from_pupae", 
    "pupae_merge_safety_reason_from_pupae"
]

suffix_drop_cols = [
    col for col in final_df.columns
    if col.endswith("_from_pupae")
]

cols_to_drop = drop_cols + suffix_drop_cols

if pupae_unmatched.empty:
    drop_cols += [
        "pupae_count",
        "Pupae_from_pupae"
        "dataset",
        "manual_inspection_reason",
        "pupae_source_row"
    ]

final_df = final_df.drop(
    columns=drop_cols,
    errors="ignore"
)

final_df["pupae_batch_folder"].value_counts()


pupae_batch_folder
VM_2026-04-29_mechano-screen-week-11_manual_f28                                                    27
VM_2026-04-29_mechano-screen-week-11_batch3_f17                                                    17
VM_2026-04-27_mechano-week-11_manual_f17                                                           17
VM_2026-04-28_mechano-screen-week-11_batch1_f10                                                    11
VM_2026-04-29_mechano-screen-week-11_batch2_f11                                                    11
VM_2026-04-30_mechano-screen-week-11_batch4_f11                                                    10
VM_2026-04-28_mechano-week-11_manual_f4                                                             4
VM_2026-04-30_mechano-week-11_manual_f1                                                             1
VM_2026-04-29_mechano-screen-week-11_manual_f28/VM_2026-04-30_mechano-screen-week-11_batch4_f11     1
Name: count, dtype: int64

In [185]:
upload_df = final_df.copy()

#upload_df = upload_df.fillna("")
upload_df = upload_df.astype(str)

worksheet.clear()

worksheet.update(
    [upload_df.columns.tolist()] + upload_df.values.tolist()
)

{'spreadsheetId': '16SmOZUOif1WM1XIff_2Hor3ve8eANdYZVRxT5jHDssA',
 'updatedRange': 'Sheet1!A1:AQ131',
 'updatedRows': 131,
 'updatedColumns': 43,
 'updatedCells': 5633}

Delete

In [ ]:
# Merge using a custom indicator name
merged = sheet_for_merge.merge(
    pupae_diag,
    on=match_cols,
    how="left",
    suffixes=("", "_from_pupae"),
    indicator="merge_status",
)

merged["Pupae"] = merged["Pupae"].replace("", pd.NA)
merged["Pupae"] = merged["Pupae"].fillna(merged["Pupae_from_pupae"])

merged["pupae_batch_folder"] = merged["pupae_batch_folder"].replace("", pd.NA)
merged["pupae_batch_folder"] = merged["pupae_batch_folder"].fillna(
    merged["pupae_batch_folder_from_pupae"]
)

merged["merge_reason"].value_counts()

In [ ]:
sheet_keys = sheet_df[match_cols].drop_duplicates()

pupae_unmatched = pupae_batch_csv.merge(
    sheet_keys,
    on=match_cols,
    how="left",
    indicator=True
)
pupae_unmatched = pupae_unmatched[
    pupae_unmatched["_merge"] == "left_only"
].drop(columns="_merge")
pupae_unmatched["manual_inspection_reason"] = "pupae row not found in Google Sheet"

for col in merged.columns:
    if col not in pupae_unmatched.columns:
        pupae_unmatched[col] = pd.NA

pupae_unmatched["manual_inspection_reason"] = "pupae row not found in Google Sheet"
pupae_unmatched = pupae_unmatched[merged.columns]

pupae_unmatched

In [ ]:
final_df = pd.concat(
    [merged, pupae_unmatched],
    ignore_index=True
)

required_update_cols = [
    "Pupae",
    "pupae_batch_folder",
    "last_edit",
]

inspection_cols = [
    "dataset",
    "pupae_count",
    "L_value",
    "manual_inspection_reason",
]

pupae_keep_cols = [
    "condition",
    "date",
    "time",
    "Pupae",
    "pupae_batch_folder",
    "dataset",
    "pupae_count",
    "L_value",
]

final_df.columns

final_df.tail(20)

In [ ]:
duplicate_cleanup_cols = [
    "staging_date_filled",
    "pupae_batch_folder_from_pupae",
    "staging_date_clean_dt",
    "L_value", 
    "merge_status",
]

final_df = final_df.drop(
    columns=[col for col in duplicate_cleanup_cols if col in sheet_df.columns],
    errors="ignore"
)


In [ ]:
#pupae_batch_csv = pd.read_csv(r"/Volumes/lab-windingm/data/instruments/behavioural_rigs/plugcamera/VM_2026-02-10-mechano-week-3_batch11_f32/pupae/predictions/pupae_counts.csv")

#for d in [pupae_batch_csv]:
    #d["pupae_batch_folder"] = pupae_batch_folder

Edit sheets manually, then save them and quit before continuing

In [ ]:
mechano_dir = Path(dirs["mechano_sheets"]) 
files = sorted(mechano_dir.glob("*.xlsx"))

dfs = []

for f in files:
    if f.suffix.lower() == ".xlsx":
        tmp = pd.read_excel(f, dtype=str)  # dtype=str avoids Excel auto-typing surprises
    else:
        tmp = pd.read_csv(f, dtype=str)

    tmp.columns = tmp.columns.str.strip()
    tmp["sheet_file"] = f.name
    tmp["sheet_name"] = f.stem
    dfs.append(tmp)

all_sheets = pd.concat(dfs, ignore_index=True)


raw = all_sheets["staging_date"]


staging_str = raw.map(
    lambda x: x.strftime("%d/%m/%Y") if isinstance(x, (pd.Timestamp,)) else ("" if pd.isna(x) else str(x).strip())
)

staging_str = staging_str.replace("", pd.NA)

all_sheets["staging_date"] = staging_str
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill()

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["time"] = pd.NA
all_sheets.loc[s.str.contains(r"\d\s*-\s*\d", na=False), "time"] = "night"
all_sheets.loc[~s.str.contains(r"\d\s*-\s*\d", na=False), "time"] = "day"

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})\s*-\s*\d{1,2}\s*/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)

all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    format="%d/%m/%Y",
    errors="coerce"
)

all_sheets



all_sheets["week"] = (
    all_sheets["sheet_file"]
    .str.extract(r"week-(\d+)", expand=False)
    .astype("Int64")
)

if all_sheets["week"].isna().any():
    bad = all_sheets.loc[all_sheets["week"].isna(), "sheet_file"].unique()
    raise ValueError(f"Could not extract week from: {bad[:5]}")

all_sheets["staging_date"] = all_sheets["staging_date"].astype("string").str.strip() #keeps blanks as missing values
all_sheets.loc[all_sheets["staging_date"] == "", "staging_date"] = pd.NA
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill() #assuming fill-down for blank rows, where order of the rows matters!

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})-\d{1,2}/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)
all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    dayfirst=True,
    errors="coerce"
)

#assume if there is a dash, assume it is overnight 
all_sheets["time"] = pd.NA
all_sheets.loc[s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "day"
all_sheets.loc[s.str.match(r"^\d{1,2}-\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "night"


In [ ]:
all_sheets["staging_date_clean_dt"].value_counts()

In [ ]:

#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")

assert_unique(all_sheets, event_key, "event_key")
assert_unique(all_sheets, plug_key, "plugcamera_per_week")

all_sheets["sheet_name"].value_counts()

target_week = 3
# adding batch folder manually (this is temporary until this part is done automatically)
batch = pupae_batch_folder

if "pupae_batch_folder" not in all_sheets.columns:
    all_sheets["pupae_batch_folder"] = pd.NA

p = all_sheets["Pupae"].astype("string").str.strip()
b = all_sheets["pupae_batch_folder"].astype("string").str.strip()

mask = (
    (all_sheets["week"] == target_week) &   # only that week
    p.notna() & (p != "") &                 # Pupae present
    (b.isna() | (b == ""))                  # batch folder missing
)

all_sheets.loc[mask, "pupae_batch_folder"] = batch


#rewriting weekly sheets manually, this will not be done when .csv files naturally merge 
#won't work if the file is open
mechano_dir = Path(dirs["mechano_sheets"])

for sheet_file, df_sheet in all_sheets.groupby("sheet_file"):
    
    if f"week-{target_week}" not in sheet_file:
        continue

    out_path = mechano_dir / sheet_file
    
    cols_to_drop = ["sheet_file", "sheet_name"]
    df_out = df_sheet.drop(columns=[c for c in cols_to_drop if c in df_sheet.columns])

    df_out.to_csv(out_path, index=False)
    print("Overwrote:", out_path)
    
all_sheets["pupae_batch_folder"].value_counts(dropna=False)

In [ ]:
mechano_dir = Path(dirs["mechano_sheets"]) 
csv_files = sorted(mechano_dir.glob("*.csv"))

dfs = []

for f in csv_files:
    tmp = pd.read_csv(f)
    tmp.columns = tmp.columns.str.strip()   
    tmp["sheet_file"] = f.name             #keeping track of where each row came from
    tmp["sheet_name"] = f.stem              
    dfs.append(tmp)

all_sheets = pd.concat(dfs, ignore_index=True)

all_sheets["week"] = (
    all_sheets["sheet_file"]
    .str.extract(r"week-(\d+)", expand=False)
    .astype("Int64")
)

if all_sheets["week"].isna().any():
    bad = all_sheets.loc[all_sheets["week"].isna(), "sheet_file"].unique()
    raise ValueError(f"Could not extract week from: {bad[:5]}")

all_sheets["staging_date"] = all_sheets["staging_date"].astype("string").str.strip() #keeps blanks as missing values
all_sheets.loc[all_sheets["staging_date"] == "", "staging_date"] = pd.NA
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill() #assuming fill-down for blank rows, where order of the rows matters!

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})-\d{1,2}/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)
all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    format="%d/%m/%Y",
    errors="coerce"
)

#assume if there is a dash, assume it is overnight 
all_sheets["time"] = pd.NA
all_sheets.loc[s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "day"
all_sheets.loc[s.str.match(r"^\d{1,2}-\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "night"

#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")

assert_unique(all_sheets, event_key, "event_key")
assert_unique(all_sheets, plug_key, "plugcamera_per_week")

all_sheets["sheet_name"].value_counts()


In [ ]:
all_sheets["week"].value_counts()

In [ ]:
pupae_batch_folder = "VM_2026-02-11_mechano-week-3_manual"
batch = pupae_batch_folder
target_week = 3

# ensure column exists
if "pupae_batch_folder" not in all_sheets.columns:
    all_sheets["pupae_batch_folder"] = pd.NA

# robust "Pupae present" check (don’t use string "nan")
pupae_num = pd.to_numeric(all_sheets["Pupae"], errors="coerce")

# robust "batch folder missing" check
bf = all_sheets["pupae_batch_folder"]
bf_str = bf.astype("string").str.strip()

missing_bf = bf.isna() | bf_str.isin(["", "nan", "NaN", "<NA>"])

mask = (all_sheets["week"] == target_week) & pupae_num.notna() & missing_bf
all_sheets.loc[mask, "pupae_batch_folder"] = batch

all_sheets["pupae_batch_folder"].value_counts()

In [ ]:
all_sheets.loc[all_sheets["week"].eq(target_week), "pupae_batch_folder"]\
    .astype("string").str.strip().value_counts(dropna=False).head(20)

bf = all_sheets["pupae_batch_folder"].astype("string").str.strip()
all_sheets.loc[bf.isin(["", "nan", "NaN", "<NA>"]), "pupae_batch_folder"] = pd.NA
all_sheets["pupae_batch_folder"].value_counts()

In [ ]:
#update master sheet (the one we show)
final_dir = Path(dirs["final-sheets"])
snap_dir = final_dir / "snapshots"
final_dir.mkdir(parents=True, exist_ok=True)
snap_dir.mkdir(parents=True, exist_ok=True)

created_at = datetime.now(timezone.utc)
stamp = created_at.strftime("%Y-%m-%dT%H%M%SZ")

all_sheets = all_sheets.copy()
all_sheets["last_edit"] = created_at.isoformat()

latest_path = final_dir / "mechano_sheets_latest.csv"
tmp_latest = final_dir / "mechano_sheets_latest.tmp.csv"

all_sheets.to_csv(tmp_latest, index=False)
tmp_latest.replace(latest_path)

# optional snapshot
#snap_path = snap_dir / f"all_sheets_{stamp}.csv"
#all_sheets.to_csv(snap_path, index=False)

# optional metadata “note”
meta = {
    "last_edit": created_at.isoformat(),
    "latest": latest_path.name,
    #"snapshot": snap_path.name,
    "rows": int(all_sheets.shape[0]),
    "cols": int(all_sheets.shape[1]),
    "inputs_dir": str(dirs["mechano_sheets"]),
}
(meta_path := snap_dir / f"mechano_sheets_{stamp}.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("Wrote latest:", latest_path)
#print("Wrote snapshot:", snap_path)
print("Wrote meta:", meta_path)


Attempting to merge automatically

In [ ]:
#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")
    
assert_unique(pupae_batch_csv, event_key, "event_key")

Not checked from here

In [ ]:
#check for duplicates in both dfs
for d in [all_sheets, pupae_batch_csv]:
    d["condition"] = d["condition"].astype("string").str.strip().str.lower()
    d["time"] = d["time"].astype("string").str.strip().str.lower()
    d["staging_date_clean_dt"] = pd.to_datetime(d["staging_date_clean_dt"], errors="coerce").dt.normalize()

pupae_batch_duplicates = pupae_batch_csv[pupae_batch_csv.duplicated(key, keep= False)].sort_values(key)
pupae_batch_clean = pupae_batch_csv.drop(index=pupae_batch_duplicates.index).copy()

pupae_batch_flagged = pupae_batch_clean.merge(
    dup_keys.assign(in_sheet_duplicates=True),
    on=key,
    how="left"
)
pupae_batch_flagged["in_sheet_duplicates"] = pupae_batch_flagged["in_sheet_duplicates"].fillna(False)

#exclude pupae batch rows giving issues
bad_rows_pupae_batch = pupae_batch_flagged[pupae_batch_flagged["condition"].isna() | pupae_batch_flagged["date"].isna() | pupae_batch_flagged["time"].isna()]
pupae_batch_drop_count_dups = pupae_batch_flagged.drop(index=bad_rows_pupae_batch.index).copy()

pupae_batch_final = pupae_batch_drop_count_dups[~pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()
manual_add_dups = pupae_batch_drop_count_dups[pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()

pupae_in = pupae_batch_final.loc[:, key + ["Pupae", "pupae_batch_folder"]].rename(
    columns={"Pupae": "_pupae_in", "pupae_batch_folder": "_pupae_folder_in"}
)

manually_add_counts_df = pd.concat([bad_rows_pupae_batch, pupae_batch_duplicates, manual_add_dups])

out_dir = Path(dirs["main"])  
out_dir.mkdir(parents=True, exist_ok=True)

#manually_add_counts_df.to_csv(
    #out_dir / f"dups_add_manually_{pupae_batch_folder}.csv",
    #index=False
#)

#manually_add_counts_df.to_csv(f"manually_add_{pupae_batch_folder}.csv")

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

if not manually_add_counts_df.empty:
    raise ValueError(
        f"{len(manually_add_counts_df)} rows need manual review (manually_add_counts_df not empty).\n"
        f"File written to: {out_dir}"
    )

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

#pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

Load weekly sheets

In [ ]:
all_sheets = Path(dirs["mechano_sheets"]) 
csv_files = sorted(all_sheets.glob("*.csv"))

dfs = []

for f in csv_files:
    tmp = pd.read_csv(f)
    tmp.columns = tmp.columns.str.strip()   
    tmp["sheet_file"] = f.name             #keeping track of where each row came from
    tmp["sheet_name"] = f.stem              
    dfs.append(tmp)

all_sheets = pd.concat(dfs, ignore_index=True)

all_sheets["week"] = (
    all_sheets["sheet_file"]
    .str.extract(r"week-(\d+)", expand=False)
    .astype("Int64")
)

if all_sheets["week"].isna().any():
    bad = all_sheets.loc[all_sheets["week"].isna(), "sheet_file"].unique()
    raise ValueError(f"Could not extract week from: {bad[:5]}")

all_sheets["staging_date"] = all_sheets["staging_date"].astype("string").str.strip() #keeps blanks as missing values
all_sheets.loc[all_sheets["staging_date"] == "", "staging_date"] = pd.NA
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill() #assuming fill-down for blank rows, where order of the rows matters!

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})-\d{1,2}/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)
all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    format="%d/%m/%Y",
    errors="coerce"
)

#assume if there is a dash, assume it is overnight 
all_sheets["time"] = pd.NA
all_sheets.loc[s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "day"
all_sheets.loc[s.str.match(r"^\d{1,2}-\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "night"

#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")

assert_unique(all_sheets, event_key, "event_key")
assert_unique(all_sheets, plug_key, "plugcamera_per_week")

In [ ]:
def safe_overwrite_csv(df: pd.DataFrame, target_path: Path) -> None:
    target_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = target_path.with_suffix(target_path.suffix + ".tmp")

    # If you don't want to write the helper columns back out, drop them here:
    out = df.drop(columns=["sheet_file", "sheet_name"], errors="ignore")

    out.to_csv(tmp_path, index=False)
    tmp_path.replace(target_path)  # atomic on same filesystem


In [ ]:
completed_sheet = pd.read_csv("260113_all_sheets.csv")
completed_sheet_missing = completed_sheet[completed_sheet["pupae"] == "n/k"]
#completed_sheet_missing.to_csv("250112_pupae_missing.csv")
completed_sheet_missing

In [ ]:
key = ["staging_date_clean_dt", "time", "condition"]
sheet_duplicates = all_sheets[all_sheets.duplicated(key, keep=False)].sort_values(key)
dup_keys = sheet_duplicates[key].drop_duplicates()

In [ ]:
sheet_flagged = all_sheets.merge(
    dup_keys.assign(in_sheet_duplicates=True), 
    on=key,
    how="left"
)

sheet_flagged["in_sheet_duplicates"] = sheet_flagged["in_sheet_duplicates"].fillna(False)
#sheet_clean = sheet_flagged.drop(index=sheet_duplicates.index).copy() #makes rows disappear

Load pupae counts batch

In [ ]:
pupae_batch_folder = "VM_2025-12-09-screen_f65_retry" #change accordingly
pupae_batch_csv = read_pupae_counts(pupae_batch_folder) #this will only work if you've "connected" to the windingm folder by opening it in your laptop
for d in [pupae_batch_csv]:
    d["pupae_batch_folder"] = pupae_batch_folder

#clean file name
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].astype(str).str.strip() #making sure it's clean strings
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.split("predictions/", n=1).str[-1]
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.replace(
    r'(\d{1,2})([-_])(\d{1,2})([-_])(day|night)\b',
    r'\1-\2_\4',
    regex=True
)
pattern = r'(?P<experiment>.+?)[_-](?P<date>\d{1,2}-\d{2})[_-](?P<time>day|night)\_'
pupae_batch_csv[["condition", "date", "time"]] = pupae_batch_csv["dataset"].str.extract(pattern, expand=True)

#extract date of experiment
year = "2025"
date_extracted = pupae_batch_csv["date"].str.extract(r'(\d{1,2}[-_]\d{1,2})', expand=False)
date_norm = date_extracted.str.replace("_", "-", regex=False)
pupae_batch_csv["staging_date_clean_dt"] = pd.to_datetime(date_norm + f"-{year}", format="%d-%m-%Y", errors="coerce")

#standardise time of day
pupae_batch_csv["time"] = pupae_batch_csv["time"].str.lower()

#calculate Pupae total
pupae_batch_csv["L_value"] = pupae_batch_csv["dataset"].str.extract(r'_L(\d+)', expand=False).astype("Int64")
pupae_batch_csv["Pupae"] = pupae_batch_csv["pupae_count"] + pupae_batch_csv["L_value"] #for clarity, the new original sheets should have a column called pupae_final rather than Pupae"

#check for duplicates in both dfs
for d in [all_sheets, pupae_batch_csv]:
    d["condition"] = d["condition"].astype("string").str.strip().str.lower()
    d["time"] = d["time"].astype("string").str.strip().str.lower()
    d["staging_date_clean_dt"] = pd.to_datetime(d["staging_date_clean_dt"], errors="coerce").dt.normalize()

pupae_batch_duplicates = pupae_batch_csv[pupae_batch_csv.duplicated(key, keep= False)].sort_values(key)
pupae_batch_clean = pupae_batch_csv.drop(index=pupae_batch_duplicates.index).copy()

pupae_batch_flagged = pupae_batch_clean.merge(
    dup_keys.assign(in_sheet_duplicates=True),
    on=key,
    how="left"
)
pupae_batch_flagged["in_sheet_duplicates"] = pupae_batch_flagged["in_sheet_duplicates"].fillna(False)

#exclude pupae batch rows giving issues
bad_rows_pupae_batch = pupae_batch_flagged[pupae_batch_flagged["condition"].isna() | pupae_batch_flagged["date"].isna() | pupae_batch_flagged["time"].isna()]
pupae_batch_drop_count_dups = pupae_batch_flagged.drop(index=bad_rows_pupae_batch.index).copy()

pupae_batch_final = pupae_batch_drop_count_dups[~pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()
manual_add_dups = pupae_batch_drop_count_dups[pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()

pupae_in = pupae_batch_final.loc[:, key + ["Pupae", "pupae_batch_folder"]].rename(
    columns={"Pupae": "_pupae_in", "pupae_batch_folder": "_pupae_folder_in"}
)

manually_add_counts_df = pd.concat([bad_rows_pupae_batch, pupae_batch_duplicates, manual_add_dups])

out_dir = Path(dirs["main"])  
out_dir.mkdir(parents=True, exist_ok=True)

#manually_add_counts_df.to_csv(
    #out_dir / f"dups_add_manually_{pupae_batch_folder}.csv",
    #index=False
#)

manually_add_counts_df.to_csv(f"manually_add_{pupae_batch_folder}.csv")

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

if not manually_add_counts_df.empty:
    raise ValueError(
        f"{len(manually_add_counts_df)} rows need manual review (manually_add_counts_df not empty).\n"
        f"File written to: {out_dir}"
    )

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

In [ ]:
pupae_batch_csv

Delete

In [ ]:
#preparing for merging, build pupae table with unique keys
merged_df = all_sheets.merge(
    pupae_in,
    on=key, 
    how='left', 
    validate="1:1"
)

target = "Pupae_final"

if target not in merged_df.columns:
    merged_df[target] = pd.NA
if "pupae_batch_folder" not in merged_df.columns:
    merged_df["pupae_batch_folder"] = pd.NA

merged_df[target] = merged_df[target].fillna(merged_df["_pupae_in"])

merged_df[target] = merged_df[target].fillna(merged_df["_pupae_in"])
merged_df["pupae_batch_folder"] = merged_df["pupae_batch_folder"].fillna(merged_df["_pupae_folder_in"])

target_col = "pupae_final"
# fill only missing
merged_df[target_col] = merged_df[target_col].fillna(merged_df["Pupae_new"])
merged_df = merged_df.drop(columns=["Pupae_new"], errors="ignore")

cols = ['pupae_total','pupae', 'Pupae_x', 'Pupae_y', "Pupae"]
populating_pupae_counter = merged_df[["experimenter", "plugcamera", "condition", "amendments", "comments",
                                      "Pupae_x", "Pupae_y", "pupae", "Pupae", "staging_date", "staging_times", "staging_date_clean_dt", "time",
                                      "sheet_name", "sheet_file", "pupae_batch_folder" ]].copy()

populating_pupae_counter[cols] = populating_pupae_counter[cols].replace('', np.nan)

pupae_cols = [c for c in populating_pupae_counter.columns if "pupae" in c.lower()]
assert pupae_cols == ["pupae_final", "pupae_batch_folder"] or set(pupae_cols) == {"pupae_final", "pupae_batch_folder"}, pupae_cols

In [ ]:
populating_pupae_counter

In [ ]:
weekly_dir = Path(dirs["weekly-sheets"])

for sheet_file, grp in populating_pupae_counter.groupby("sheet_file", sort=False):
    safe_overwrite_csv(grp, weekly_dir / sheet_file)

In [ ]:
#do checks on the relevant files
which_sheets = merged_df[merged_df["pupae_batch_folder"].notna()]
which_sheets["sheet_file"].value_counts()

In [ ]:
merged_keys = merged_df.loc[merged_df["pupae_batch_folder"].notna(), key].drop_duplicates()

pupae_keys = pupae_batch_drop_count_dups[key].drop_duplicates()

missing_keys = pupae_keys.merge(merged_keys, on=key, how="left", indicator=True)
missing_keys = missing_keys[missing_keys["_merge"] == "left_only"].drop(columns="_merge")

missing_pupae_rows = pupae_batch_drop_count_dups.merge(missing_keys, on=key, how="inner")

print("Pupae rows that did not end up in merged_df:")
print(missing_pupae_rows[key + ["dataset", "pupae_count", "L_value", "Pupae", "pupae_batch_folder"]]
      .to_string(index=False))

In [ ]:
out_dir = Path(dirs["main"])  
out_dir.mkdir(parents=True, exist_ok=True)

missing_pupae_rows.to_csv(
    out_dir / f"add_manually_{pupae_batch_folder}.csv",
    index=False
)